Ising with nearest-neighbor interactions

Assign energies to each state.

States: 
- closed, adjacent closed (zero energy)
    - 1 adjacent up (x2?)
    - 2 adjacent up
- open, adjacent closed ($\epsilon$ energy)
    - 1 adjacent up
    - 2 adjacent up

All closed is minimum energy. Low energy is higher probability? At what temp does it work out at?

Partition function of one state:

$$
Z = \sum_s e^{-\epsilon_s/\tau} \\
\epsilon_{0,0} = 0 \\
\epsilon_{0,1} = a \\ %not sure what these are yet
\epsilon_{0,2} = b \\
\epsilon_{1,0} = \epsilon + c\\
\epsilon_{1,1} = \epsilon + d \\
\epsilon_{1,2} = \epsilon \\

P(s,n) = \frac{e^{-\epsilon_{s,n}/\tau}}{e^{-\epsilon_{0,n}} + e^{-\epsilon_{1,n}}} \\
$$

Really there is only two states, but state energy updates each round. We'll use a markov chain to monitor these.

Edges: use open vals



Can we induct it down? break into 2 pieces? Course graining makes fewer degrees of freedom by "averaging", certain parameters can make this useful. "Real Space Renormalization". 

Assumptions for our current system:

Canonical ensemble: any two states can happen. Needs large time steps (seconds vs microsecond interactions)

Transformation from full partition into product of single particles? Need to do some googling first. Someone else has done it clever before. Some kind of linear combination, some kind of forier? Waves on a string decouples into modes. Could be fun.



In [ ]:
import numpy as np
from numpy import exp
from scipy.constants import k
import seaborn as sns
import matplotlib.pyplot as plt

#Temp = 20 # °C
#tau = k*(Temp + 273.15)
tau=5
a = 1
b = a*2
d = 1
c = d*2
epsilon = 5

Z_table = np.array([
    [1, exp(-a/tau), exp(-b/tau)],
    [exp(-(epsilon+c)/tau), exp(-(epsilon+d)/tau), exp(epsilon/tau)]
])

#Z_table = np.array([
#    [1000, 5, 1],
#    [1, 1, 100]
#])

n = 100 #chain length
chain = np.ones(n, dtype=int)
neighbors = np.zeros(n, dtype=int)
chain_list = [chain.copy()]

for i in range(100):

    
    neighbors[1:-1] = chain[2:] + chain[:-2]
    neighbors[0] = 1 + chain[1]
    neighbors[-1] = 1 + chain[-2]

    probability_threshold = Z_table[chain, neighbors] / Z_table.sum(axis=0)[neighbors] # probability to stayin current state
    if i % 100 == 0:
        print(probability_threshold.mean())
    state_change = np.random.rand(n)>probability_threshold
    chain[state_change] = 1-chain[state_change]

    #if i == 20:
    #    chain[200:210] = 0

    chain_list.append(chain.copy())

plt.figure(figsize=(10,10))
sns.heatmap(np.array(chain_list), xticklabels=False, yticklabels=False, cbar=False)

print(chain)
print(neighbors)
plt.title(f'N={n} Chain Evolution Mural')

The above code has an issue: every single site changes simultaneously. This makes nucleation (and stability) difficult since simultaneous switching can occur and reshuffle the entire plot. Let's try attempting to update each site one at a time, that way there is an opportunity for neighbors to interact within each row.

In [ ]:
# Try to use random update order instead of simultaneous update -> breaks up checkerboard
rng = np.random.default_rng()

Z_table = np.array([
    [10000, 1, 1],
    [1, 2, 5]
])

n = 500 #chain length
random_order = np.arange(1,n-1) # boundary conditions
chain = np.ones(n+2, dtype=int)
chain_list = [chain.copy()]

for i in range(1000): # time steps
    rng.shuffle(random_order)
    
    for i in random_order:
        neighbors = chain[i-1] + chain[i+1]
        probability_threshold = Z_table[chain[i], neighbors] / Z_table.sum(axis=0)[neighbors] #p of staying in current state
        chain[i] = chain[i] ^ (probability_threshold<np.random.rand())

    chain[0] = chain[1] #edge is just the same as neighbor state
    chain[-1] = chain[-2]

    chain_list.append(chain.copy())

plt.figure(figsize=(20,40))
sns.heatmap(np.array(chain_list), xticklabels=False, yticklabels=False, cbar=False)

This is cool, but no set of parameters has shown nucleation. It would be nice to determine if interesting dynamics can occur before running a simulation of parameters, so we can skip running all-single-state or all-mixed simulations with no actual changes over time.

We can brute force calculate the boltzmann factors for each state and partition function of smaller chains and check if they are flat or not.

In [ ]:
from numpy import exp

def get_state_energy(chain, e_table, loop = False):
    '''
    Used to find probability of state occuring once Z is calculated
    edge condition is currently open
    '''
    neighbors = np.copy(chain)*0
    neighbors[1:-1] = chain[2:] + chain[:-2]
    if loop == False:
        neighbors[0] = 1 + chain[1]
        neighbors[-1] = 1 + chain[-2]
    else:
        neighbors[0] = chain[1] + chain[-1]
        neighbors[-1] = chain[-2] + chain[0] 
    
    energies = e_table[chain, neighbors]
    total_energy = energies.sum()
    return total_energy

In [ ]:
tau=2
ground = 0
a = 2
b = 5
c = 5
d = 2
epsilon = 0

N = 18
neighbors = np.zeros(N, dtype=int)

e_table = np.array([
    [ground, ground+a, ground+b],
    [epsilon+c, epsilon+d, epsilon]
])

In [ ]:
# Use pandas to get the probability of all states
# Z from previous cell
import pandas as pd

chain_list = [np.array([int(bit) for bit in f'{i:0{N}b}']) for i in range(2**N)]
chain_energy = [get_state_energy(chain, e_table, loop=True) for chain in chain_list]
boltzmann_factors = [exp(-energy/tau) for energy in chain_energy]
Z = sum(boltzmann_factors)
chain_prob = np.array(boltzmann_factors)/Z
open_states = [sum(chain) for chain in chain_list]

chain_dict = {'state':chain_list, 'energy':chain_energy, 'probability':chain_prob, 'open_states':open_states}

All_States = pd.DataFrame(chain_dict)

#All_States['probability'].sum()
number_open = []
for i in range(N):
    state_probability = sum(All_States.loc[All_States['open_states'] == i]['probability'])
    number_open.append((i, state_probability))

In [ ]:
prob_array = np.array(number_open) 
plt.axvline(N/2, color='red')
plt.title(f'length N={N} chain with nearest-neigbor interactions')
plt.plot(prob_array[:,0], prob_array[:, 1])
plt.xlabel('excited states (n)')
plt.ylabel('probability')
plt.grid()